In [4]:
# Подход: эмбеддинги через sentence-transformers, модель intfloat/multilingual-e5-small.
!pip install sentence-transformers
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = '/kaggle/input/datasets/zlatta/dataset'  # папка с файлами

TRAIN_PATH = os.path.join(DATA_DIR, 'train.parquet')
QUERIES_PATH = os.path.join(DATA_DIR, 'benchmark_queries.parquet')
ITEMS_PATH = os.path.join(DATA_DIR, 'benchmark_items.parquet')

# Куда сохранять результаты
WORK_DIR = '/kaggle/working'
OUTPUT_PATH = os.path.join(WORK_DIR, 'answer.csv')
CACHE_DIR = os.path.join(WORK_DIR, 'embeddings_cache')  # папка для промежуточных эмбеддингов
os.makedirs(CACHE_DIR, exist_ok=True)


print('Загружаем данные...')
queries = pd.read_parquet(QUERIES_PATH)
items = pd.read_parquet(ITEMS_PATH)

print(f'Запросов: {len(queries)}')
print(f'Объявлений: {len(items)}')

# Модель e5 требует префиксов: "query:" для запросов, "passage:" для документов.

def prepare_item_text(row):
    # Склеиваем заголовок, описание и параметры объявления в одну строку.
    parts = [
        str(row.get('item_title_raw') or ''),
        str(row.get('item_description_raw') or ''),
        str(row.get('item_infm_params_text') or ''),
    ]
    return 'passage: ' + ' '.join(p for p in parts if p).strip()

def prepare_query_text(row):
    # Склеиваем текст запроса и фильтры поиска.
    parts = [
        str(row.get('search_query') or ''),
        str(row.get('search_infm_params_text') or ''),
    ]
    return 'query: ' + ' '.join(p for p in parts if p).strip()

print('Готовим тексты...')
items['text'] = items.apply(prepare_item_text, axis=1)
queries['text'] = queries.apply(prepare_query_text, axis=1)

print('Загружаем модель...')
model = SentenceTransformer('intfloat/multilingual-e5-small')

# Эмбеддинги объявлений считаем батчами и сохраняем после каждого батча.
# Если Colab отвалится, при следующем запуске продолжим с того же места.
ITEMS_EMB_PATH = os.path.join(CACHE_DIR, 'items_embeddings.npy')

BATCH_SIZE = 512  # сколько объявлений обрабатываем за один шаг
texts = items['text'].tolist()
total = len(texts)

# Проверяем, сколько объявлений уже обработано в предыдущих запусках.
if os.path.exists(ITEMS_EMB_PATH):
    item_embeddings = np.load(ITEMS_EMB_PATH)
    done = len(item_embeddings)
    print(f'Нашли сохранённые эмбеддинги: {done} из {total}')
else:
    item_embeddings = np.zeros((0, 384), dtype=np.float32)  # 384 — размерность e5-small
    done = 0

# Обрабатываем оставшиеся батчи
while done < total:
    end = min(done + BATCH_SIZE, total)
    print(f'Обрабатываем объявления {done}–{end} из {total}...')

    batch = texts[done:end]
    batch_emb = model.encode(
        batch,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # Приклеиваем новые эмбеддинги к накопленным
    item_embeddings = np.vstack([item_embeddings, batch_emb])
    done = end

    # Сохраняем после каждого батча
    np.save(ITEMS_EMB_PATH, item_embeddings)
    print(f'Прогресс сохранён: {done} из {total}')

print(f'Готово. Размер матрицы эмбеддингов: {item_embeddings.shape}')

print('Считаем эмбеддинги запросов...')
query_embeddings = model.encode(
    queries['text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Поиск топ-50 кандидатов для каждого запроса.
print('Ищем кандидатов для каждого запроса...')
predictions = []

for i, row in tqdm(queries.iterrows(), total=len(queries)):
    # Эмбеддинг текущего запроса
    q_emb = query_embeddings[i:i+1]  # форма (1, dim)

    # Косинусное сходство со всеми объявлениями
    # Так как эмбеддинги нормализованы, косинус = скалярное произведение
    scores = cosine_similarity(q_emb, item_embeddings)[0]

    # Фильтр по категории: оставляем только объявления той же категории
    # Если в нужной категории меньше 50 объявлений, фильтр не применяем.
    cat_mask = items['item_category_id'] == row['search_category']
    if cat_mask.sum() >= 50:
        # Там, где категория не совпадает, ставим -1
        scores = np.where(cat_mask, scores, -1.0)

    # Бонус за совпадение локации (небольшой, чтобы не перебивать текст)
    loc_mask = items['item_location_id'] == row['search_location_id']
    if loc_mask.sum() > 0:
        scores = scores + 0.1 * loc_mask

    # Берём топ-50 по скору
    top_idx = scores.argsort()[-50:][::-1]
    top_ids = items.iloc[top_idx]['item_id'].tolist()
    predictions.append(top_ids)

print('Сохраняем answer.csv...')
answer = pd.DataFrame({
    'query_id': queries['query_id'].astype(str),
    'answer': [' '.join(str(x) for x in p) for p in predictions],
})

answer.to_csv(OUTPUT_PATH, index=False)
print(f'Готово: {OUTPUT_PATH}')
print(f'Строк в файле: {len(answer)}')

Загружаем данные...
Запросов: 2452
Объявлений: 189212
Готовим тексты...
Загружаем модель...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Нашли сохранённые эмбеддинги: 44544 из 189212
Обрабатываем объявления 44544–45056 из 189212...
Прогресс сохранён: 45056 из 189212
Обрабатываем объявления 45056–45568 из 189212...
Прогресс сохранён: 45568 из 189212
Обрабатываем объявления 45568–46080 из 189212...
Прогресс сохранён: 46080 из 189212
Обрабатываем объявления 46080–46592 из 189212...
Прогресс сохранён: 46592 из 189212
Обрабатываем объявления 46592–47104 из 189212...
Прогресс сохранён: 47104 из 189212
Обрабатываем объявления 47104–47616 из 189212...
Прогресс сохранён: 47616 из 189212
Обрабатываем объявления 47616–48128 из 189212...
Прогресс сохранён: 48128 из 189212
Обрабатываем объявления 48128–48640 из 189212...
Прогресс сохранён: 48640 из 189212
Обрабатываем объявления 48640–49152 из 189212...
Прогресс сохранён: 49152 из 189212
Обрабатываем объявления 49152–49664 из 189212...
Прогресс сохранён: 49664 из 189212
Обрабатываем объявления 49664–50176 из 189212...
Прогресс сохранён: 50176 из 189212
Обрабатываем объявления 50176–

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Ищем кандидатов для каждого запроса...


100%|██████████| 2452/2452 [11:26<00:00,  3.57it/s]

Сохраняем answer.csv...
Готово: /kaggle/working/answer.csv
Строк в файле: 2452
